# Figuras ilustrativas de la tesis

Notebook **auxiliar y no numerado**: genera las figuras conceptuales y de apoyo para el documento escrito (introducción, marco teórico y metodología). No forma parte del pipeline de análisis; solo produce imágenes en el estilo visual unificado de la tesis (paleta **SLP_RGB**, español, 300 dpi).

Las figuras se guardan en **imagenes/ilustrativas/** y se copian luego a **latex/recursos/img/**.

| Figura | Archivo | Ubicación en la tesis |
|---|---|---|
| Composición global (donut) | **intro_estatico_donut.png** | Fig. 1.1A (Introducción) |
| Hipnograma representativo | **intro_dinamico_hipno.png** | Fig. 1.1B (Introducción) |
| Modelo Skip-Gram | **skipgram_sueno.png** | Fig. 2.2 (Marco teórico) |
| Flujo general del proyecto | **pipeline_general.png** | Metodología |
| Ventaneo deslizante (stride 1) | **ventana_deslizante.png** | Metodología |
| LSTM desplegada | **lstm_desplegada.png** | Metodología |

## Configuración: estilo, paleta y utilidades de dibujo


In [1]:
"""Figuras ilustrativas de la tesis, en el estilo limpio de la Figura 2.1."""
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle

RAIZ = Path("..").resolve()
OUT = RAIZ / "imagenes/ilustrativas"
OUT.mkdir(parents=True, exist_ok=True)
CSV = RAIZ / "dataset/epocas_unificado.csv"

FASES = {0: "Vigilia", 1: "S1", 2: "S2", 3: "SWS", 4: "REM"}
SLP_RGB = [
    (230/255, 230/255, 250/255),  # Vigilia
    (173/255, 216/255, 230/255),  # S1
    (173/255, 255/255,  47/255),  # S2
    (255/255, 127/255,  80/255),  # SWS
    ( 65/255, 105/255, 225/255),  # REM
]
# paleta del estilo Figura 2.1
FIRE = (0.60, 0.06, 0.06)   # rojo oscuro titulos
BLUE = (0.00, 0.00, 0.85)   # azul lineas
LBLUE = (0.86, 0.91, 0.99)  # azul claro relleno cajas
BLACK = (0.05, 0.05, 0.05)
GRIS = (0.45, 0.45, 0.45)

plt.rcParams.update({
    "font.size": 13, "savefig.dpi": 300, "savefig.bbox": "tight",
    "font.family": "DejaVu Sans",
})


def _txtbox(ax, x, y, w, h, txt, fc=LBLUE, ec=BLUE, tc=BLACK, fs=12, bold=True, rnd=0.08, lw=2.0):
    ax.add_patch(FancyBboxPatch((x-w/2, y-h/2), w, h,
                 boxstyle=f"round,pad=0.01,rounding_size={rnd}",
                 fc=fc, ec=ec, lw=lw, zorder=2))
    ax.text(x, y, txt, ha="center", va="center", fontsize=fs,
            fontweight="bold" if bold else "normal", color=tc, zorder=3)


def _arrow(ax, p0, p1, color=BLACK, lw=2.2, style="-|>", mut=18):
    ax.add_patch(FancyArrowPatch(p0, p1, arrowstyle=style, mutation_scale=mut,
                 color=color, lw=lw, zorder=1, shrinkA=3, shrinkB=3))


# ---------------------------------------------------------------- Fig 1.1A donut

## Fig. 1.1A — Composición global del sueño (donut)

Porcentajes reales de cada fase sobre las 127 noches sanas del dataset unificado.


In [2]:
def fig_donut():
    df = pd.read_csv(CSV)
    s = df[df["fase_num"].between(0, 4)]["fase_num"].value_counts(normalize=True).sort_index() * 100
    vals = [s.get(i, 0.0) for i in range(5)]
    fig, ax = plt.subplots(figsize=(7, 7))
    wedges, _ = ax.pie(vals, colors=SLP_RGB, startangle=90, counterclock=False,
                       wedgeprops=dict(width=0.42, edgecolor="white", linewidth=2))
    for w, i, v in zip(wedges, range(5), vals):
        ang = np.deg2rad((w.theta1 + w.theta2) / 2)
        x, y = np.cos(ang), np.sin(ang)
        ax.text(0.79*x, 0.79*y, f"{FASES[i]}\n{v:.1f}%", ha="center", va="center",
                fontsize=12.5, fontweight="bold", color="white" if i == 4 else BLACK)
    ax.text(0, 0, "Composición\nglobal\n(127 noches)", ha="center", va="center",
            fontsize=13.5, fontweight="bold", color=FIRE)
    ax.set(aspect="equal")
    fig.savefig(OUT / "intro_estatico_donut.png")
    plt.close(fig)


# ---------------------------------------------------------------- Fig 1.1B dinamico (esquemático limpio)

fig_donut()
print("OK:", OUT / "fig_donut")

OK: imagenes/ilustrativas/fig_donut


## Fig. 1.1B — Hipnograma de un participante sano representativo

Mismo estilo y paleta que el notebook 3 (sin título ni barra de color), participante **CAP_S12_N1**.

In [3]:
def fig_dinamico():
    # Hipnograma estilo notebook 3 (mismos colores SLP_RGB que la intro estática),
    # sin título superior ni barra de color inferior.
    df = pd.read_csv(CSV)
    g = df[(df["paciente"] == "CAP_S12_N1") & (df["fase_num"].between(0, 4))].sort_values("epoca").copy()
    g["h"] = g["epoca"] / 120.0
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(g["h"], g["fase_num"], color="k", linewidth=0.9, zorder=2)
    ax.scatter(g["h"], g["fase_num"], s=12, color=[SLP_RGB[i] for i in g["fase_num"]], zorder=3)
    ax.set_yticks(list(FASES.keys()))
    ax.set_yticklabels(list(FASES.values()), fontsize=12)
    ax.set_ylim(-0.1, 4.1)
    ax.set_xlabel("Tiempo (horas)", fontsize=12.5)
    ax.grid(color="gray", linestyle="--", linewidth=0.4, axis="y")
    fig.tight_layout()
    fig.savefig(OUT / "intro_dinamico_hipno.png")
    plt.close(fig)


# ---------------------------------------------------------------- Fig 2.2 Skip-Gram

fig_dinamico()
print("OK:", OUT / "fig_dinamico")

OK: imagenes/ilustrativas/fig_dinamico


## Fig. 2.2 — Modelo Skip-Gram aplicado al sueño

Esquema limpio (estilo de la Fig. 2.1): a partir del token central se predicen las fases del contexto.


In [4]:
def fig_skipgram():
    fig, ax = plt.subplots(figsize=(10.5, 5.6))
    ax.set_xlim(0, 10.5); ax.set_ylim(0, 6.2); ax.axis("off")
    ax.text(5.25, 5.85, "Modelo Skip-Gram aplicado al sueño", ha="center",
            fontsize=15.5, fontweight="bold", color=FIRE)
    # entrada
    _txtbox(ax, 1.5, 3.1, 2.0, 1.1, "Token central\n2-2-2-2-2-2", fc=LBLUE, ec=BLUE, fs=11.5)
    ax.text(1.5, 1.95, "(entrada)", ha="center", fontsize=10.5, color=GRIS, style="italic")
    # proyeccion
    _txtbox(ax, 4.6, 3.1, 1.9, 1.3, "Embedding\n32-dim", fc="white", ec=BLUE, fs=12)
    ax.text(4.6, 1.95, "(proyección)", ha="center", fontsize=10.5, color=GRIS, style="italic")
    # salidas contexto
    ys = [4.9, 3.9, 2.3, 1.3]
    labs = ["fase $t-2$", "fase $t-1$", "fase $t+1$", "fase $t+2$"]
    for y, lab in zip(ys, labs):
        _txtbox(ax, 8.6, y, 1.9, 0.78, lab, fc=LBLUE, ec=BLUE, fs=11)
        _arrow(ax, (5.55, 3.1), (7.6, y), color=GRIS, lw=1.8, mut=15)
    ax.text(8.6, 5.55, "contexto", ha="center", fontsize=10.5, color=GRIS, style="italic")
    _arrow(ax, (2.5, 3.1), (3.65, 3.1), color=FIRE, lw=2.6, mut=20)
    fig.savefig(OUT / "skipgram_sueno.png")
    plt.close(fig)


# ---------------------------------------------------------------- Pipeline

fig_skipgram()
print("OK:", OUT / "fig_skipgram")

OK: imagenes/ilustrativas/fig_skipgram


## Flujo general del proyecto

De las tres fuentes a la unificación, la tokenización y los modelos de análisis.


In [5]:
def fig_pipeline():
    fig, ax = plt.subplots(figsize=(12, 6.4))
    ax.set_xlim(0, 12); ax.set_ylim(0, 6.6); ax.axis("off")
    ax.text(6, 6.25, "Flujo general del proyecto", ha="center", fontsize=15.5,
            fontweight="bold", color=FIRE)
    fuentes = [(1.6, 5.2, "CAP (16)\nS3+S4 → SWS"),
               (1.6, 3.2, "Sleep-EDF (82)\nedad ≤ 65"),
               (1.6, 1.2, "Scoring (29)")]
    for x, y, t in fuentes:
        _txtbox(ax, x, y, 2.5, 1.15, t, fc="white", ec=BLUE, fs=11)
    _txtbox(ax, 4.9, 3.2, 2.6, 1.6, "Unificación\n127 pacientes\n112 809 épocas\n5 fases (0–4)",
            fc=LBLUE, ec=BLUE, fs=11)
    for _, y, _ in fuentes:
        _arrow(ax, (2.9, y), (3.58, 3.2), color=GRIS, lw=1.8, mut=15)
    _txtbox(ax, 7.9, 3.2, 2.2, 1.3, "Tokenización\nventana N=6\nstride 1", fc=LBLUE, ec=BLUE, fs=11)
    _arrow(ax, (6.22, 3.2), (6.78, 3.2), color=FIRE, lw=2.4, mut=18)
    modelos = [(10.7, 4.6, "Cadenas de Markov"),
               (10.7, 3.2, "Word2Vec\n(embeddings)"),
               (10.7, 1.8, "LSTM\n(lenguaje del sueño)")]
    for x, y, t in modelos:
        _txtbox(ax, x, y, 2.4, 1.05, t, fc=LBLUE, ec=BLUE, fs=10.5)
        _arrow(ax, (9.04, 3.2), (9.48, y), color=GRIS, lw=1.6, mut=14)
    fig.savefig(OUT / "pipeline_general.png")
    plt.close(fig)


# ---------------------------------------------------------------- Ventana deslizante (rediseñada)

fig_pipeline()
print("OK:", OUT / "fig_pipeline")

OK: imagenes/ilustrativas/fig_pipeline


## Ventaneo deslizante (stride = 1)

Migración escalonada: cada ventana de 6 épocas se desplaza una época; tokens consecutivos comparten 5 de 6 épocas.


In [6]:
def fig_ventana():
    # estilo "migración/rejilla del deslizador": la secuencia completa arriba y,
    # debajo, las ventanas de 6 épocas apiladas y desplazadas 1 época (escalera).
    sub = [0, 1, 2, 2, 3, 3, 3, 2, 2]   # Vigilia S1 S2 S2 SWS SWS SWS S2 S2
    n = len(sub)
    cw, ch = 1.0, 0.8
    nwin = n - 6 + 1   # ventanas de tamaño 6
    fig, ax = plt.subplots(figsize=(11.5, 5.8))
    ax.set_xlim(-0.4, n + 3.2)
    ax.set_ylim(-0.55 - (nwin-1)*0.98 - 0.22, 1.7)
    ax.axis("off")
    ax.text(n/2, 1.55, "Secuencia de épocas (30 s cada una)", ha="center",
            fontsize=14, fontweight="bold", color=FIRE)
    # fila de referencia (secuencia completa)
    for i, f in enumerate(sub):
        ax.add_patch(Rectangle((i+0.06, 0.55), cw*0.88, ch, fc=SLP_RGB[f], ec="white", lw=2.5, zorder=2))
        ax.text(i+0.5, 0.95, FASES[f], ha="center", va="center", fontsize=10,
                fontweight="bold", color="white" if f == 4 else BLACK, zorder=3)
    # ventanas apiladas en escalera
    marco = [(0.80, 0.10, 0.10), (0.10, 0.45, 0.15), (0.10, 0.25, 0.70), (0.55, 0.35, 0.05)]
    for k in range(nwin):
        yb = -0.55 - k*0.98
        c = marco[k % len(marco)]
        for j in range(6):
            f = sub[k+j]
            ax.add_patch(Rectangle((k+j+0.06, yb), cw*0.88, ch, fc=SLP_RGB[f],
                         ec="white", lw=2.0, zorder=2))
            ax.text(k+j+0.5, yb+0.40, str(f), ha="center", va="center", fontsize=9.5,
                    fontweight="bold", color="white" if f == 4 else BLACK, zorder=3)
        # marco de color de la ventana
        ax.add_patch(FancyBboxPatch((k+0.02, yb-0.04), 6.0-0.04, ch+0.08,
                     boxstyle="round,pad=0.005,rounding_size=0.05",
                     fc="none", ec=c, lw=2.8, zorder=4))
        tok = "-".join(str(sub[k+j]) for j in range(6))
        ax.text(n+0.35, yb+0.40, f"Token {k+1}: {tok}", ha="left", va="center",
                fontsize=12, fontweight="bold", color=c)
    fig.savefig(OUT / "ventana_deslizante.png")
    plt.close(fig)


# ---------------------------------------------------------------- LSTM desplegada (rediseñada)

fig_ventana()
print("OK:", OUT / "fig_ventana")

OK: imagenes/ilustrativas/fig_ventana


## LSTM desplegada

La red lee la secuencia y predice la fase siguiente en cada paso, propagando el estado de memoria.


In [7]:
def fig_lstm():
    pasos_in = [0, 2, 3, 2]      # fases de entrada (ejemplo)
    pasos_out = [2, 3, 2, 4]     # fase predicha siguiente
    xs = [1.7, 4.4, 7.1, 9.8]
    fig, ax = plt.subplots(figsize=(11.5, 5.8))
    ax.set_xlim(0, 11.5); ax.set_ylim(0, 6.0); ax.axis("off")
    ax.text(5.75, 5.65, "LSTM desplegada: lee la secuencia y predice la fase siguiente paso a paso",
            ha="center", fontsize=13.5, fontweight="bold", color=FIRE)
    for i, x in enumerate(xs):
        fin, fout = pasos_in[i], pasos_out[i]
        # entrada (celda de fase)
        ax.add_patch(Rectangle((x-0.55, 0.45), 1.1, 0.7, fc=SLP_RGB[fin], ec=BLACK, lw=1.6, zorder=3))
        ax.text(x, 0.80, FASES[fin], ha="center", va="center", fontsize=10.5,
                fontweight="bold", color="white" if fin == 4 else BLACK, zorder=4)
        ax.text(x, 0.05, "entrada", ha="center", fontsize=9.5, color=GRIS, style="italic")
        # celda LSTM
        _txtbox(ax, x, 2.7, 1.5, 1.0, "LSTM", fc=LBLUE, ec=BLUE, fs=13)
        # salida (prediccion)
        ax.add_patch(Rectangle((x-0.55, 4.25), 1.1, 0.7, fc=SLP_RGB[fout], ec=BLACK, lw=1.6, zorder=3))
        ax.text(x, 4.60, FASES[fout], ha="center", va="center", fontsize=10.5,
                fontweight="bold", color="white" if fout == 4 else BLACK, zorder=4)
        ax.text(x, 5.05, "predicción", ha="center", fontsize=9.5, color=GRIS, style="italic")
        _arrow(ax, (x, 1.18), (x, 2.18), color=BLACK, lw=2.0, mut=16)
        _arrow(ax, (x, 3.22), (x, 4.20), color=BLACK, lw=2.0, mut=16)
        if i < len(xs)-1:
            _arrow(ax, (x+0.78, 2.7), (xs[i+1]-0.78, 2.7), color=FIRE, lw=2.4, mut=18)
            ax.text((x+xs[i+1])/2, 2.95, "memoria $h_t$", ha="center", va="bottom",
                    fontsize=9.5, color=FIRE, style="italic")
    fig.savefig(OUT / "lstm_desplegada.png")
    plt.close(fig)

fig_lstm()
print("OK:", OUT / "fig_lstm")

OK: imagenes/ilustrativas/fig_lstm


## Copiar las figuras a la carpeta de LaTeX


In [8]:
import shutil
DEST = RAIZ / 'latex' / 'recursos' / 'img'
for png in sorted(OUT.glob('*.png')):
    shutil.copy(png, DEST / png.name)
    print('copiado ->', png.name)

copiado -> intro_dinamico_hipno.png
copiado -> intro_estatico_donut.png
copiado -> lstm_desplegada.png
copiado -> pipeline_general.png
copiado -> skipgram_sueno.png
copiado -> ventana_deslizante.png
